# Experiment 2 — Feynman Extrapolation (n=30) — Kaggle Multithreading Edition

**HypatiaX Research · RF-09 full-budget run**

Uses **`parallelism='multithreading'`** — Julia threads in-process, no worker subprocesses.  
Switched from `multiprocessing` after `Distributed.ProcessExitedException` on equation 2  
(Julia worker killed by Kaggle; `remotecall_fetch` deadlock on dead worker).

| Setting | Value |
|---|---|
| Timeout | 1100 s/equation |
| Populations | 30 |
| Population size | 33 |
| Iterations | 1 000 |
| Max size | 30 |
| Parsimony | 0.01 |
| Parallelism | **multithreading** (Julia threads, in-process — no Distributed) |
| Runtime estimate | ~12-24 h on 4-vCPU CPU notebook |

> **Recommended runtime:** Settings → Accelerator → **None** (CPU-only = 4 vCPUs).  
> GPU notebooks give only 2 CPUs — worse for PySR which is CPU-bound.

> **Why not multiprocessing?**  
> `parallelism='multiprocessing'` uses Julia's `Distributed` stdlib.  
> On Kaggle with Julia 1.11.x, worker subprocesses (PID 3+) get killed by  
> the container OS (memory pressure / signal handling). When worker 3 dies,  
> `remotecall_fetch` blocks on `take!(::Distributed.RemoteValue)` then raises  
> `ProcessExitedException(3)`, crashing the entire PySR fit.  
> `multithreading` runs all populations in Julia threads inside the main process —  
> no subprocess to kill, no `Distributed` stack, immune to this failure mode.

> **Setup order:** Run Cell 1 FIRST, then Cell 2 (installs Julia + PySR).  
> **API key:** Use Kaggle Secrets (notebook Settings → Add-ons → Secrets).


## 1 · Environment setup

**Run this cell FIRST.**

In [1]:
# CELL 1 -- Run this BEFORE installing PySR
import os, multiprocessing
n_cores = multiprocessing.cpu_count()
print(f'Kaggle CPU cores available: {n_cores}')
# CPU-only runtime (Settings -> Accelerator -> None) gives 4 vCPUs.
# GPU notebooks (T4/P100) give only 2 CPUs -- worse for PySR.
os.environ['JULIA_NUM_THREADS']               = str(n_cores)
os.environ['JULIA_EXCLUSIVE']                 = '0'
# Prevent juliacall / PyTorch signal-handler collision (segfault guard).
os.environ['PYTHON_JULIACALL_HANDLE_SIGNALS'] = 'yes'
# LLM model used for any API calls in this notebook.
os.environ['LLM_MODEL'] = 'claude-sonnet-4-20250514'
print(f"JULIA_NUM_THREADS set to {os.environ['JULIA_NUM_THREADS']} OK")
print(f"LLM_MODEL = {os.environ['LLM_MODEL']}")
print()
print('NOTE: parallelism=multithreading will use all JULIA_NUM_THREADS.')
print('      multiprocessing is disabled — Distributed.ProcessExitedException')
print('      kills runs on Kaggle/Julia 1.11 when worker subprocesses are OOM-killed.')


Kaggle CPU cores available: 4
JULIA_NUM_THREADS set to 4 OK
LLM_MODEL = claude-sonnet-4-20250514


## 2 · Install Julia + dependencies

> Julia is not pre-installed on Kaggle. Download takes ~3-5 min.


In [2]:
import subprocess, sys, os
# Julia not pre-installed on Kaggle -- download it first
JULIA_VERSION = '1.11.4'
JULIA_MINOR   = '1.11'
print(f'Downloading Julia {JULIA_VERSION}...')
tarball = f'julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
url = (
    f'https://julialang-s3.julialang.org/bin/linux/x64/{JULIA_MINOR}/'
    f'{tarball}'
)
subprocess.run(['wget', '-q', url], check=True)
subprocess.run(['tar', '-xzf', tarball], check=True)
julia_bin = f'/kaggle/working/julia-{JULIA_VERSION}/bin'
os.environ['PATH'] = julia_bin + ':' + os.environ.get('PATH', '')
print(f'Julia {JULIA_VERSION} installed OK')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pysr', 'scikit-learn', 'scipy', 'numpy', 'pandas'], check=True)
import pysr
print(f'PySR version: {pysr.__version__}')
result = subprocess.run(['julia', '-e', 'println(Threads.nthreads())'],
                        capture_output=True, text=True, timeout=120)
print(f'Julia threads: {result.stdout.strip()}')


Julia 1.11.4 installed OK
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 8.6 MB/s eta 0:00:00
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/pysr/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Using Julia 1.11.4 at /kaggle/working/julia-1.11.4/bin/julia
[juliapkg] Using Julia project at /root/.julia/environments/pyjuliapkg
[juliapkg] Writing Project.toml:
           | [deps]
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46603ac705cb"
           | Seri

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed pixi_jll ───────────────── v0.41.3+0
   Installed ScientificTypesBase ────── v3.1.0
   Installed MicroMamba ─────────────── v0.1.15
   Installed Tricks ─────────────────── v0.1.13
   Installed Adapt ──────────────────── v4.5.2
   Installed JSON ───────────────────── v1.5.2
   Installed DynamicExpressions ─────── v1.10.4
   Installed PythonCall ─────────────── v0.9.26
   Installed PositiveFactorizations ─── v0.2.4
   Installed StatisticalTraits ──────── v3.5.0
   Installed MLJModelInterface ──────── v1.11.1
   Installed ADTypes ────────────────── v1.22.0
   Installed Preferences ────────────── v1.5.2
   Installed OpenSSL_jll ────────────── v3.0.20+0
   Installed Pidfile ────────────────── v1.3.0
   Installed Parsers ────────────────── v2.8.4
   Installed ProgressMeter ──────────── v1.10.2
   Installed micromamba_jll ─────────── v2.3.1+0
   Installed NLSolversBase ──────────── v7.10.

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
PySR version: 1.5.10
Julia threads: 4


## 3 · Imports

In [3]:
import os, time, json, warnings, inspect
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from scipy import stats as scipy_stats
warnings.filterwarnings('ignore')
try:
    from pysr import PySRRegressor
    PYSR_AVAILABLE = True
    print('PySR loaded OK')
except ImportError:
    PYSR_AVAILABLE = False
    print('ERROR: pysr not installed')
_PYSR_VALID_PARAMS = None

PySR loaded OK


## 4 · API key

Settings → Add-ons → Secrets → add `ANTHROPIC_API_KEY`.

In [4]:
# Kaggle Secrets: notebook Settings -> Add-ons -> Secrets
# -> Add ANTHROPIC_API_KEY -> Attach to notebook
import os
try:
    from kaggle_secrets import UserSecretsClient
    ANTHROPIC_API_KEY = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')
    print('API key loaded from Kaggle Secrets OK')
except Exception as e:
    print(f'Kaggle Secrets unavailable: {e}')
    ANTHROPIC_API_KEY = ''  # paste key here only as last resort
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY or ''
if ANTHROPIC_API_KEY and ANTHROPIC_API_KEY.startswith('sk-ant-'):
    print('API key set OK')
else:
    print('Key missing -- add via Kaggle Secrets')

API key loaded from Kaggle Secrets OK
API key set OK


## 5 · Run configuration

In [5]:
import os
OUTPUT_DIR = '/kaggle/working'
CFG = dict(
    timeout=1100,       # feynman_pysr_seconds from repro.yaml
    populations=30,
    iterations=1000,
    n_equations=30,
    seed=42,
    output_json=os.path.join(OUTPUT_DIR, 'exp2_feynman_extrap_kaggle.json'),
    nn_only=False,
)
print('Config:', CFG)
print(f"Checkpoints -> {CFG['output_json']}")
print('Files in /kaggle/working appear in the Output tab.')


Config: {'timeout': 1100, 'populations': 30, 'iterations': 1000, 'n_equations': 30, 'seed': 42, 'output_json': '/kaggle/working/exp2_feynman_extrap_kaggle.json', 'nn_only': False}
Checkpoints -> /kaggle/working/exp2_feynman_extrap_kaggle.json
Files in /kaggle/working appear in the Output tab.


## 6 · 30-equation Feynman dataset

AI Feynman dataset (Udrescu & Tegmark 2020), 11 physics domains.

In [ ]:
FEYNMAN_30 = [
    # --- Mechanics (5) ---
    {
        'id': 'I.6.2', 'name': 'Gaussian', 'domain': 'Mechanics',
        'vars': ['sigma', 'x', 'mu'],
        'ranges': [(0.5, 2.0), (-5.0, 5.0), (-1.0, 1.0)],
        'extrap_multiplier': 2.0,
        'fn': lambda sigma, x, mu: (
            np.exp(-((x - mu)**2) / (2 * sigma**2)) / (np.sqrt(2 * np.pi) * sigma)
        ),
    },
    {
        'id': 'I.12.2', 'name': 'Coulomb Force', 'domain': 'Mechanics',
        'vars': ['q1', 'q2', 'r'],
        'ranges': [(1e-9, 1e-6), (1e-9, 1e-6), (0.01, 1.0)],
        'extrap_multiplier': 2.0,
        'fn': lambda q1, q2, r: 8.99e9 * q1 * q2 / r**2,
    },
    {
        'id': 'I.15.10', 'name': 'Relativistic momentum', 'domain': 'Mechanics',
        'vars': ['m', 'v', 'c'],
        'ranges': [(0.1, 10.0), (0.0, 0.9), (1.0, 1.0)],
        'extrap_multiplier': 1.5,
        'fn': lambda m, v, c: m * v / np.sqrt(1 - (v / c)**2),
        'note': 'v/c must stay < 1; extrap range capped at 0.95c',
    },
    {
        'id': 'I.34.8', 'name': 'Doppler shift', 'domain': 'Mechanics',
        'vars': ['omega', 'v', 'c'],
        'ranges': [(1e6, 1e9), (0.0, 0.5), (1.0, 1.0)],
        'extrap_multiplier': 1.5,
        'fn': lambda omega, v, c: omega * (1 + v / c) / np.sqrt(1 - (v / c)**2),
    },
    {
        'id': 'I.50.26', 'name': 'Harmonic oscillator', 'domain': 'Mechanics',
        'vars': ['x1', 'omega', 't', 'alpha'],
        'ranges': [(0.1, 2.0), (0.5, 5.0), (0.0, 10.0), (0.01, 0.1)],
        'extrap_multiplier': 2.0,
        'fn': lambda x1, omega, t, alpha: x1 * (np.cos(omega * t) + alpha * t),
    },
    # --- Thermodynamics (2) ---
    {
        'id': 'I.12.4', 'name': 'Electric potential energy', 'domain': 'Thermodynamics',
        'vars': ['q1', 'q2', 'r'],
        'ranges': [(1e-9, 1e-6), (1e-9, 1e-6), (0.01, 1.0)],
        'extrap_multiplier': 2.0,
        'fn': lambda q1, q2, r: 8.99e9 * q1 * q2 / r,
    },
    {
        'id': 'I.34.27', 'name': 'Energy of photon', 'domain': 'Thermodynamics',
        'vars': ['omega'],
        'ranges': [(1e12, 1e15)],
        'extrap_multiplier': 2.0,
        'fn': lambda omega: 1.055e-34 * omega,
    },
    {
        'id': 'II.34.29b', 'name': 'Magnetization', 'domain': 'Thermodynamics',
        'vars': ['n', 'g', 'Jz', 'mu_B', 'B', 'kb', 'T'],
        'ranges': [
            (1e22, 1e24), (1.0, 3.0), (0.5, 2.0),
            (9.27e-24, 9.27e-24), (0.01, 10.0),
            (1.38e-23, 1.38e-23), (100, 1000)
        ],
        'extrap_multiplier': 2.0,
        'fn': lambda n, g, Jz, mu_B, B, kb, T: (
            n * g * Jz * mu_B * np.tanh(g * Jz * mu_B * B / (kb * T))
        ),
    },
    # --- Optics (3) ---
    {
        'id': 'I.34.14', 'name': 'Relativistic Doppler', 'domain': 'Optics',
        'vars': ['omega_0', 'v', 'c'],
        'ranges': [(1e12, 1e15), (0.0, 0.5), (3e8, 3e8)],
        'extrap_multiplier': 1.5,
        'fn': lambda omega_0, v, c: omega_0 * np.sqrt(1 - (v / c)**2) / (1 - v / c),
    },
    {
        'id': 'II.2.42', 'name': 'Heat conduction', 'domain': 'Optics',
        'vars': ['kappa', 'T1', 'T2', 'd'],
        'ranges': [(0.1, 10.0), (200, 400), (400, 600), (0.01, 1.0)],
        'extrap_multiplier': 2.0,
        'fn': lambda kappa, T1, T2, d: kappa * (T2 - T1) / d,
    },
    {
        'id': 'I.26.2', 'name': "Snell's law", 'domain': 'Optics',
        'vars': ['n1', 'theta1', 'n2'],
        'ranges': [(1.0, 1.5), (0.1, 1.2), (1.3, 2.0)],
        'extrap_multiplier': 1.5,
        'fn': lambda n1, theta1, n2: np.arcsin(n1 * np.sin(theta1) / n2),
    },
    # --- Electromagnetism (4) ---
    {
        'id': 'II.11.3', 'name': 'Polarization', 'domain': 'Electromagnetism',
        'vars': ['n_0', 'alpha', 'Ef', 'T'],
        'ranges': [(1e22, 1e24), (1e-30, 1e-29), (1e4, 1e6), (100, 1000)],
        'extrap_multiplier': 2.0,
        'fn': lambda n_0, alpha, Ef, T: (
            n_0 * alpha * Ef / (1 - n_0 * alpha / (3 * 8.85e-12))
        ),
    },
    {
        'id': 'I.18.12', 'name': 'Torque', 'domain': 'Electromagnetism',
        'vars': ['r', 'F', 'theta'],
        'ranges': [(0.1, 5.0), (1.0, 100.0), (0.0, np.pi)],
        'extrap_multiplier': 2.0,
        'fn': lambda r, F, theta: r * F * np.sin(theta),
    },
    {
        'id': 'I.37.4', 'name': 'Interference intensity', 'domain': 'Electromagnetism',
        'vars': ['I1', 'I2', 'delta'],
        'ranges': [(0.1, 10.0), (0.1, 10.0), (0.0, 2 * np.pi)],
        'extrap_multiplier': 1.5,
        'fn': lambda I1, I2, delta: I1 + I2 + 2 * np.sqrt(I1 * I2) * np.cos(delta),
    },
    {
        'id': 'II.11.27', 'name': 'Polarizability', 'domain': 'Electromagnetism',
        'vars': ['n', 'alpha', 'eps'],
        'ranges': [(1e22, 1e24), (1e-30, 1e-29), (1e-12, 1e-10)],
        'extrap_multiplier': 2.0,
        'fn': lambda n, alpha, eps: n * alpha / (1 - n * alpha / (3 * eps)),
    },
    {
        'id': 'I.41.16', 'name': 'Planck radiation', 'domain': 'Electromagnetism',
        'vars': ['omega', 'T'],
        'ranges': [(1e12, 1e13), (100, 1000)],
        'extrap_multiplier': 2.0,
        'fn': lambda omega, T: (
            1.055e-34 * omega**3 / (np.pi**2 * (3e8)**2)
            / (np.exp(1.055e-34 * omega / (1.38e-23 * T)) - 1)
        ),
    },
    # --- Quantum mechanics (3) ---
    {
        'id': 'custom.hbar_omega', 'name': 'Photon energy', 'domain': 'Quantum',
        'vars': ['h_bar', 'omega'],
        'ranges': [(1.055e-34, 1.055e-34), (1e13, 1e16)],
        'extrap_multiplier': 2.0,
        'fn': lambda h_bar, omega: h_bar * omega,
        'note': 'E=hbar*omega (photon energy). ID corrected: FSRD I.34.1 is relativistic Doppler; '
                'this formula matches FSRD I.34.27 but h_bar is fixed here as a constant input. '
                'Relabelled custom.hbar_omega to avoid ID collision.',
    },
    {
        'id': 'II.34.2a', 'name': 'Magnetic moment', 'domain': 'Quantum',
        'vars': ['q', 'v', 'r'],
        'ranges': [(1e-19, 1e-18), (1e3, 1e6), (1e-10, 1e-8)],
        'extrap_multiplier': 2.0,
        'fn': lambda q, v, r: q * v * r / 2,
    },
    {
        'id': 'III.4.32', 'name': 'Bose-Einstein', 'domain': 'Quantum',
        'vars': ['h_bar', 'omega', 'kb', 'T'],
        'ranges': [(1.055e-34, 1.055e-34), (1e12, 1e13), (1.38e-23, 1.38e-23), (100, 1000)],
        'extrap_multiplier': 2.0,
        'fn': lambda h_bar, omega, kb, T: 1 / (np.exp(h_bar * omega / (kb * T)) - 1),
    },
    # --- Gravitation (2) ---
    {
        'id': 'I.12.1', 'name': 'Gravitational force', 'domain': 'Gravitation',
        'vars': ['m1', 'm2', 'r'],
        'ranges': [(1e10, 1e12), (1e10, 1e12), (1e6, 1e8)],
        'extrap_multiplier': 2.0,
        'fn': lambda m1, m2, r: 6.674e-11 * m1 * m2 / r**2,
        'note': 'FSRD I.12.1 is gravitational FORCE F=Gm1m2/r^2 (fixed from erroneous 1/r potential). '
                'Large dynamic range — expected hard case for SR.',
    },
    {
        'id': 'custom.kepler_period', 'name': 'Orbital period (Kepler)', 'domain': 'Gravitation',
        # NOTE: I.50.26b does not exist in the FSRD-100 or any official release.
        # Kepler's 3rd law has no assigned FSRD ID. Relabelled custom.kepler_period.  # original (erroneous) id was 'I.50.26b'
        'vars': ['r', 'M'],
        'ranges': [(1e6, 1e9), (1e20, 1e30)],
        'extrap_multiplier': 2.0,
        'fn': lambda r, M: 2 * np.pi * np.sqrt(r**3 / (6.674e-11 * M)),
    },
    # --- Fluid mechanics (2) ---
    {
        'id': 'II.11.17', 'name': 'Dielectric constant', 'domain': 'Fluid',
        'vars': ['n', 'alpha', 'eps0'],
        'ranges': [(1e22, 1e24), (1e-30, 1e-29), (8.85e-12, 8.85e-12)],
        'extrap_multiplier': 2.0,
        'fn': lambda n, alpha, eps0: 1 + n * alpha / eps0 / (1 - n * alpha / (3 * eps0)),
    },
    {
        'id': 'I.30.5', 'name': 'Diffraction', 'domain': 'Fluid',
        'vars': ['n', 'd', 'theta'],
        # BUG FIX: n clamped to [1,2]; at n=5,d=1e-6 -> n*lambda/d=2.5 > 1
        # causing arcsin domain error in both training and extrap ranges.
        'ranges': [(1, 2), (1e-6, 1e-4), (0.01, 0.4)],
        'extrap_multiplier': 1.5,
        'fn': lambda n, d, theta: (
            np.arcsin(np.clip(n * 5e-7 / d + np.sin(theta), -1.0, 1.0))
        ),
        'note': 'n clamped [1,2] to keep arcsin argument in [-1,1]; lambda=500nm fixed',
    },
    # --- Waves (2) ---
    {
        'id': 'I.29.4', 'name': 'Wave superposition', 'domain': 'Waves',
        'vars': ['r', 'omega', 't', 'v'],
        'ranges': [(0.1, 10.0), (1.0, 10.0), (0.0, 5.0), (1.0, 5.0)],
        'extrap_multiplier': 2.0,
        'fn': lambda r, omega, t, v: np.sin(omega * (t - r / v)) / r,
    },
    {
        'id': 'I.48.2', 'name': 'de Broglie wavelength', 'domain': 'Waves',
        'vars': ['m', 'v'],
        'ranges': [(9.1e-31, 1e-27), (1e3, 1e6)],
        'extrap_multiplier': 2.0,
        'fn': lambda m, v: 6.626e-34 / (m * v),
    },
    # --- Special relativity (2) ---
    {
        'id': 'I.34.27b', 'name': 'Time dilation', 'domain': 'Relativity',
        # NOTE: I.34.27b is from the 2022 SRSD extension (Kamienny et al.), not the original FSRD-100.
        'vars': ['t', 'v', 'c'],
        'ranges': [(1.0, 100.0), (0.0, 0.8), (1.0, 1.0)],
        'extrap_multiplier': 1.5,
        'fn': lambda t, v, c: t / np.sqrt(1 - (v / c)**2),
    },
    {
        'id': 'I.34.10', 'name': 'Lorentz factor', 'domain': 'Relativity',
        'vars': ['v', 'c'],
        'ranges': [(0.0, 0.7), (1.0, 1.0)],
        'extrap_multiplier': 1.3,
        'fn': lambda v, c: 1 / np.sqrt(1 - (v / c)**2),
        'note': 'v/c extrap capped at 0.95 to avoid singularity',
    },
    # --- Atomic physics (2) ---
    {
        'id': 'II.2.4', 'name': 'Coulomb potential', 'domain': 'Atomic',
        'vars': ['q1', 'q2', 'r', 'eps'],
        'ranges': [(1e-19, 1e-18), (1e-19, 1e-18), (1e-10, 1e-8), (8.85e-12, 8.85e-12)],
        'extrap_multiplier': 2.0,
        'fn': lambda q1, q2, r, eps: q1 * q2 / (4 * np.pi * eps * r),
    },
    {
        'id': 'I.43.31', 'name': 'Diffusion coefficient', 'domain': 'Atomic',
        'vars': ['mob', 'T'],
        'ranges': [(1e-8, 1e-5), (200, 400)],
        'extrap_multiplier': 2.0,
        'fn': lambda mob, T: mob * 1.38e-23 * T,
    },
    # --- Nuclear (1) ---
    {
        'id': 'II.34.29a', 'name': 'Larmor frequency', 'domain': 'Nuclear',
        'vars': ['q', 'B', 'm'],
        'ranges': [(1e-19, 1e-18), (0.01, 10.0), (1e-30, 1e-27)],
        'extrap_multiplier': 2.0,
        'fn': lambda q, B, m: q * B / (2 * m),
    },
]

assert len(FEYNMAN_30) == 30, f'Expected 30 equations, got {len(FEYNMAN_30)}'
print(f'{len(FEYNMAN_30)} equations loaded across {len(set(e["domain"] for e in FEYNMAN_30))} domains')

## 7 · Helper functions

In [7]:
def generate_data(eq, N=200, noise_level=0.05, seed=42):
    rng = np.random.RandomState(seed)
    n_vars = len(eq['vars'])
    extrap_ranges = []
    for i, (lo, hi) in enumerate(eq['ranges']):
        vname = eq['vars'][i]
        if lo == hi:
            extrap_ranges.append((lo, hi))
        elif vname == 'v' and hi <= 1.0:
            extrap_ranges.append((hi, min(hi * eq['extrap_multiplier'], 0.95)))
        else:
            extrap_ranges.append((hi, hi * eq['extrap_multiplier']))
    X_tr = np.column_stack([rng.uniform(lo, hi, N) for lo, hi in eq['ranges']])
    try:
        y_tr = eq['fn'](*[X_tr[:, i] for i in range(n_vars)])
        valid = np.isfinite(y_tr)
        X_tr, y_tr = X_tr[valid], y_tr[valid]
    except Exception as e:
        return None, None, None, None, str(e)
    noise_std = noise_level * np.std(y_tr)
    y_tr_noisy = y_tr + rng.normal(0, noise_std, len(y_tr))
    X_train, _, y_train, _ = train_test_split(X_tr, y_tr_noisy, test_size=0.2, random_state=seed)
    X_ext = np.column_stack([rng.uniform(lo, hi, 100) for lo, hi in extrap_ranges])
    try:
        y_ext = eq['fn'](*[X_ext[:, i] for i in range(n_vars)])
        valid = np.isfinite(y_ext)
        X_ext, y_ext = X_ext[valid], y_ext[valid]
    except Exception:
        X_ext, y_ext = None, None
    return X_train, y_train, X_ext, y_ext, None

def safe_r2(y_true, y_pred):
    if y_true is None or y_pred is None or len(y_true) == 0:
        return None
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    thresh = max(1e-10 * (np.max(np.abs(y_true)) ** 2) * len(y_true), 1e-300)
    if ss_tot < thresh:
        return 1.0 if ss_res < 1e-20 else 0.0
    return 1 - ss_res / ss_tot

print('Helper functions defined')

Helper functions defined


## 8 · Neural network baseline (64-32-1 MLP)

In [8]:
def run_nn(X_train, y_train, X_ext, y_ext, seed=42):
    """MLP baseline — matches paper architecture (64-32-1, ReLU, Adam)."""
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_s = scaler_X.fit_transform(X_train)
    y_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()

    nn = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        learning_rate_init=0.01,
        max_iter=200,
        random_state=seed,
    )
    t0 = time.time()
    nn.fit(X_s, y_s)
    elapsed = time.time() - t0

    train_r2 = safe_r2(
        y_train,
        scaler_y.inverse_transform(nn.predict(X_s).reshape(-1, 1)).ravel()
    )
    extrap_r2 = None
    if X_ext is not None:
        y_pred_ext = scaler_y.inverse_transform(
            nn.predict(scaler_X.transform(X_ext)).reshape(-1, 1)
        ).ravel()
        extrap_r2 = safe_r2(y_ext, y_pred_ext)

    return {'train_r2': train_r2, 'extrap_r2': extrap_r2, 'time_s': elapsed}


print('NN baseline defined')

NN baseline defined


## 9 · HypatiaX (PySR) — multithreading

> Uses `parallelism='multithreading'` — Julia threads in-process across all CPUs.  
> **Why not multiprocessing?**  Julia 1.11 + Kaggle: worker subprocesses (PID 3+) get  
> OOM-killed by the container; `remotecall_fetch` then hangs on a dead worker and  
> raises `Distributed.ProcessExitedException(3)`, crashing the PySR fit.  
> Multithreading avoids all subprocesses — populations run in Julia threads inside  
> the main process, immune to this failure.  
> PySR config aligned with `repro.yaml` (population_size=33, maxsize=30, parsimony=0.01).


In [ ]:
def make_pysr(seed=42, niterations=1000, timeout_secs=1100, populations=30):
    """
    PySRRegressor configured for PySR 1.x (installed on Kaggle: 1.5.x).

    parallelism='multithreading':
      Uses Julia threads (JULIA_NUM_THREADS) inside the main process.
      No worker subprocesses — immune to Distributed.ProcessExitedException
      which kills multiprocessing runs on Kaggle/Julia 1.11 (worker OOM-killed,
      remotecall_fetch blocks on take!(::Distributed.RemoteValue) then crashes).

    PySR 1.x API notes:
      REMOVED: deterministic, batching, batch_size, procs, multithreading (kwarg)
               (parallelism='multithreading' is the 1.x way to set threading)
      CHANGED: mutation_weights now expects MutationWeights namedtuple, not dict
               — passing a dict silently crashes inside __init__ and returns
               time=0s from the except block. Removed entirely; defaults are fine.
      KEPT:    parallelism, timeout_in_seconds, tournament_selection_n,
               crossover_probability, random_state, populations, population_size,
               maxsize, parsimony, verbosity, progress

    Reproducibility: random_state=seed gives deterministic results in 1.x.
    """
    if not PYSR_AVAILABLE:
        raise RuntimeError('PySR not installed')

    global _PYSR_VALID_PARAMS
    if _PYSR_VALID_PARAMS is None:
        _PYSR_VALID_PARAMS = set(inspect.signature(PySRRegressor.__init__).parameters.keys())
    valid = _PYSR_VALID_PARAMS

    # Core params — valid in all PySR versions
    kwargs = dict(
        niterations     = niterations,
        populations     = populations,
        population_size = 33,     # repro.yaml: pysr.population_size
        maxsize         = 30,     # repro.yaml: pysr.maxsize
        parsimony       = 0.01,   # repro.yaml: pysr.parsimony
        binary_operators  = ['+', '-', '*', '/'],
        unary_operators   = ['exp', 'log', 'sin', 'cos', 'sqrt'],
        random_state    = seed,   # replaces deterministic=True in PySR 1.x
        verbosity       = 0,
        progress        = False,
    )

    # timeout_in_seconds — valid in all versions
    if 'timeout_in_seconds' in valid:
        kwargs['timeout_in_seconds'] = timeout_secs

    # parallelism='multithreading' — uses JULIA_NUM_THREADS, no subprocesses.
    # MUST NOT use 'multiprocessing' on Kaggle/Julia 1.11:
    #   Distributed worker PIDs get OOM-killed → ProcessExitedException(3)
    #   → remotecall_fetch hangs → entire fit crashes.
    if 'parallelism' in valid:
        kwargs['parallelism'] = 'multithreading'

    # tournament_selection_n — valid in 1.x
    if 'tournament_selection_n' in valid:
        kwargs['tournament_selection_n'] = 3

    # crossover_probability — valid in 1.x
    if 'crossover_probability' in valid:
        kwargs['crossover_probability'] = 0.9

    # NOTE: Do NOT pass mutation_weights as a dict — PySR 1.x expects a
    # MutationWeights namedtuple; passing a dict raises TypeError inside
    # __init__ which is caught by run_hypatia's except block, returning
    # train_r2=None, extrap_r2=None, time_s=0 silently. Default weights fine.

    # NOTE: Do NOT pass deterministic or batching — removed in PySR 1.x.

    return PySRRegressor(**kwargs)


def run_hypatia(X_train, y_train, X_ext, y_ext, seed=42,
                niterations=1000, timeout_secs=1100, populations=30):
    if not PYSR_AVAILABLE:
        return {'train_r2': None, 'extrap_r2': None, 'time_s': 0,
                'error': 'pysr_not_installed'}
    try:
        model = make_pysr(seed=seed, niterations=niterations,
                          timeout_secs=timeout_secs, populations=populations)
    except Exception as e:
        print(f"  [make_pysr ERROR] {type(e).__name__}: {e}")
        return {'train_r2': None, 'extrap_r2': None, 'time_s': 0, 'error': str(e)}

    t0 = time.time()
    try:
        model.fit(X_train, y_train)
        elapsed = time.time() - t0
    except Exception as e:
        elapsed = time.time() - t0
        print(f"  [model.fit ERROR] {type(e).__name__}: {e}")
        return {'train_r2': None, 'extrap_r2': None, 'time_s': elapsed, 'error': str(e)}

    try:
        train_r2  = safe_r2(y_train, model.predict(X_train))
        extrap_r2 = safe_r2(y_ext, model.predict(X_ext)) if X_ext is not None else None
        best_expr = str(model.sympy())
    except Exception as e:
        print(f"  [model.predict/sympy ERROR] {type(e).__name__}: {e}")
        train_r2  = extrap_r2 = None
        best_expr = f'eval_error: {e}'

    return {'train_r2': train_r2, 'extrap_r2': extrap_r2,
            'time_s': elapsed, 'best_expression': best_expr}


print('HypatiaX (PySR 1.x multithreading) wrapper defined')
print('parallelism=multithreading: Julia threads in-process, no Distributed subprocesses.')


## 10 · Main loop — run all 30 equations

Checkpointed to `/kaggle/working/` after every equation.  
Re-run after session restart — completed equations are skipped.

In [ ]:
equations = FEYNMAN_30[:CFG['n_equations']]

# Resume: load any existing checkpoint
all_results = {}
if os.path.exists(CFG['output_json']):
    with open(CFG['output_json']) as f:
        all_results = json.load(f)
    print(f"▶ Resume: loaded {len(all_results)} completed result(s) from {CFG['output_json']}")
else:
    print('Starting fresh run')

print('=' * 65)
print(f"EXPERIMENT 2: FEYNMAN EXTRAPOLATION (n={len(equations)})")
print(f"timeout={CFG['timeout']}s  populations={CFG['populations']}  "
      f"iterations={CFG['iterations']}")
print('=' * 65)

for i, eq in enumerate(equations):
    if eq['id'] in all_results:
        print(f"[{i+1:2d}/30] ⏭  {eq['id']}: {eq['name']} — already done, skipping")
        continue

    eq_seed = CFG['seed'] + i * 7
    print(f"\n[{i+1:2d}/30] {eq['id']}: {eq['name']} [{eq['domain']}]")
    if eq.get('note'):
        print(f"  NOTE: {eq['note']}")

    X_train, y_train, X_ext, y_ext, data_err = generate_data(
        eq, N=200, noise_level=0.05, seed=eq_seed
    )
    if data_err:
        print(f'  DATA ERROR: {data_err}')
        all_results[eq['id']] = {'name': eq['name'], 'domain': eq['domain'], 'error': data_err}
        continue

    record = {'name': eq['name'], 'domain': eq['domain'], 'note': eq.get('note', '')}

    # ── NN baseline ──────────────────────────────────────────────────────────
    print('  Running NN...')
    record['nn'] = run_nn(X_train, y_train, X_ext, y_ext, seed=eq_seed)
    _nn = record['nn']['extrap_r2']
    print(f"  NN:  train_R²={record['nn']['train_r2']:.4f}  "
          f"extrap_R²={f'{_nn:.4f}' if _nn is not None else 'N/A'}")

    # ── HypatiaX (PySR) ──────────────────────────────────────────────────────
    if not CFG['nn_only']:
        print('  Running HypatiaX (PySR)...')
        record['hypatia'] = run_hypatia(
            X_train, y_train, X_ext, y_ext,
            seed=eq_seed,
            niterations=CFG['iterations'],
            timeout_secs=CFG['timeout'],
            populations=CFG['populations'],
        )
        h = record['hypatia']
        _ht, _he = h.get('train_r2'), h.get('extrap_r2')
        print(f"  HYP: train_R²={f'{_ht:.4f}' if _ht is not None else 'N/A'}  "
              f"extrap_R²={f'{_he:.4f}' if _he is not None else 'N/A'}  "
              f"time={h.get('time_s', 0):.0f}s")
        if h.get('error'):
            print(f"  ERR:  {h['error'][:120]}")
        if h.get('best_expression') and not h.get('error'):
            print(f"  EXPR: {h['best_expression']}")

    all_results[eq['id']] = record

    # Checkpoint save — enables resume after disconnect
    with open(CFG['output_json'], 'w') as f:
        json.dump(all_results, f, indent=2)

print('\n' + '=' * 65)
print('All equations completed.')
print(f"Results saved → {CFG['output_json']}")

## 11 · Statistical analysis

In [ ]:
hyp_ext_all = [
    r.get('hypatia', {}).get('extrap_r2')
    for r in all_results.values()
    if r.get('hypatia', {}).get('extrap_r2') is not None
]
nn_ext_all = [
    r.get('nn', {}).get('extrap_r2')
    for r in all_results.values()
    if r.get('nn', {}).get('extrap_r2') is not None
]

successes = {
    eid: r for eid, r in all_results.items()
    if (r.get('hypatia', {}).get('extrap_r2') or -999) > 0.99
}
hyp_ext_succ = [r['hypatia']['extrap_r2'] for r in successes.values()]
nn_ext_succ  = [r.get('nn', {}).get('extrap_r2')
                for r in successes.values()
                if r.get('nn', {}).get('extrap_r2') is not None]

n_hyp = len(hyp_ext_all)
n_succ = len(hyp_ext_succ)

print('=' * 65)
print('RESULTS SUMMARY')
print('=' * 65)
print(f"HypatiaX  n={n_hyp}  "
      f"mean={np.mean(hyp_ext_all):.4f}  "
      f"median={np.median(hyp_ext_all):.4f}  "
      f"success(>0.99)={n_succ}/{n_hyp} ({n_succ/n_hyp*100:.1f}%)")
print(f"NN        n={len(nn_ext_all)}  "
      f"mean={np.mean(nn_ext_all):.4f}  "
      f"median={np.median(nn_ext_all):.4f}")

print()
# Test 1 — all n=30
if len(hyp_ext_all) >= 5 and len(nn_ext_all) >= 5:
    u1, p1 = scipy_stats.mannwhitneyu(hyp_ext_all, nn_ext_all, alternative='greater')
    sig1 = 'SIGNIFICANT ✓' if p1 < 0.05 else 'not significant'
    print(f"Test 1 — all n=30:           U={u1:.0f},  p={p1:.4e}  [{sig1}]")

# Test 2 — successes only
if len(hyp_ext_succ) >= 3 and len(nn_ext_succ) >= 3:
    u2, p2 = scipy_stats.mannwhitneyu(hyp_ext_succ, nn_ext_succ, alternative='greater')
    sig2 = 'SIGNIFICANT ✓' if p2 < 0.05 else 'not significant'
    print(f"Test 2 — successes (n={n_succ}):  "
          f"U={u2:.0f},  p={p2:.4e}  [{sig2}]")
else:
    print(f'Test 2 — successes: only {n_succ} success(es) — not enough for MW test')

print()
print('Successful equations:')
for eid, r in successes.items():
    he = r['hypatia']['extrap_r2']
    expr = r['hypatia'].get('best_expression', 'N/A')
    print(f"  {eid:12s}  extrap_R²={he:.4f}  expr: {expr}")

## 12 · Results table

In [ ]:
import pandas as pd

rows = []
for eq in FEYNMAN_30:
    eid = eq['id']
    r = all_results.get(eid, {})
    h = r.get('hypatia', {})
    nn = r.get('nn', {})
    he = h.get('extrap_r2')
    result = '✓ Win' if (he is not None and he > 0.99) else ('✗ Fail' if he is not None else '—')
    rows.append({
        'ID':           eid,
        'Name':         eq['name'],
        'Domain':       eq['domain'],
        'H train R²':   f"{h.get('train_r2'):.3f}" if h.get('train_r2') is not None else '—',
        'H extrap R²':  f"{he:.3f}" if he is not None else '—',
        'NN extrap R²': f"{nn.get('extrap_r2'):.3f}" if nn.get('extrap_r2') is not None else '—',
        'Result':       result,
    })

df = pd.DataFrame(rows)

def colour_result(val):
    if '✓' in str(val):
        return 'background-color: #d4edda; color: #155724; font-weight: bold'
    if '✗' in str(val):
        return 'background-color: #f8d7da; color: #721c24'
    return ''

df.style.applymap(colour_result, subset=['Result'])

## 13 · LaTeX table for §6

In [ ]:
CLIP_LO = -15.0

def _cell(v, is_extrap=False):
    if v is None:
        return '---'
    if is_extrap:
        if v > 0.99:
            return r'\PASS{' + f'{v:.3f}' + '}'
        elif v < 0:
            if v < CLIP_LO:
                return r'\FAIL{$\ll{-}100$}'
            return r'\FAIL{' + f'${v:.3f}$' + '}'
    return f'{v:.3f}'


lines = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Full 30-equation Feynman extrapolation results: '
    r'HypatiaX vs.\ Neural Network at $2\times$ training range. '
    r'Extrap $R^2$ below $-15$ marked as catastrophic failures. '
    r'\PASS{Green}: $R^2>0.99$; \FAIL{Red}: $R^2<0$.}',
    r'\label{tab:feynman30_extrap}',
    r'\small',
    r'\setlength{\tabcolsep}{5pt}',
    r'\begin{tabular}{llccc}',
    r'\toprule',
    r'\textbf{Equation} & \textbf{Domain} & '
    r'\textbf{Hyp Train $R^2$} & \textbf{Hyp Extrap $R^2$} & '
    r'\textbf{NN Extrap $R^2$} \\',
    r'\midrule',
]

hyp_ext_vals, nn_ext_vals = [], []
hyp_ext_clipped, nn_ext_clipped = [], []
hyp_successes = nn_successes = 0
n_total = 0
shade = False

for eq_id, res in all_results.items():
    h  = res.get('hypatia', {})
    nn = res.get('nn', {})
    he, ne, ht = h.get('extrap_r2'), nn.get('extrap_r2'), h.get('train_r2')

    if he is not None:
        hyp_ext_vals.append(he)
        hyp_ext_clipped.append(max(he, CLIP_LO))
        if he > 0.99: hyp_successes += 1
    if ne is not None:
        nn_ext_vals.append(ne)
        nn_ext_clipped.append(max(ne, CLIP_LO))
        if ne > 0.99: nn_successes += 1
    n_total += 1

    row_prefix = r'\rowcolor{lightgray}' + '\n' if shade else ''
    shade = not shade
    lines.append(
        row_prefix +
        f"{res['name']} & {res['domain']} & "
        f"{_cell(ht)} & {_cell(he, True)} & {_cell(ne, True)} \\\\"
    )

raw_hyp  = np.mean(hyp_ext_vals)    if hyp_ext_vals    else float('nan')
raw_nn   = np.mean(nn_ext_vals)     if nn_ext_vals     else float('nan')
clip_hyp = np.mean(hyp_ext_clipped) if hyp_ext_clipped else float('nan')
clip_nn  = np.mean(nn_ext_clipped)  if nn_ext_clipped  else float('nan')

lines += [
    r'\midrule', r'\midrule',
    f'\\textbf{{Mean extrap $R^2$ (raw)}} & --- & --- & ${raw_hyp:.2e}$ & ${raw_nn:.2e}$ \\\\',
    f'\\textbf{{Mean extrap $R^2$ (clip $-15$)}} & --- & --- & {clip_hyp:.3f} & {clip_nn:.3f} \\\\',
    f'\\textbf{{Success}} ($R^2 > 0.99$) & --- & --- & '
    r'\PASS{' + f'{hyp_successes}/{n_total} ({hyp_successes/n_total*100:.1f}\\%)' + '} & '
    r'\FAIL{' + f'{nn_successes}/{n_total} ({nn_successes/n_total*100:.1f}\\%)' + '} \\\\',
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table}',
]

latex = '\n'.join(lines)
tex_path = CFG['output_json'].replace('.json', '_table.tex')
with open(tex_path, 'w') as f:
    f.write(latex)

print(latex)
print(f'\nLaTeX table saved → {tex_path}')

## 14 · Export all formats

All files land in `/kaggle/working/` — download via the Output tab.

In [ ]:
import os, json, zipfile, shutil
from datetime import datetime
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_DIR = f'hypatia_exp2_{timestamp}'
os.makedirs(EXPORT_DIR, exist_ok=True)
print(f'Export folder: {EXPORT_DIR}/')

In [ ]:
# 1 · JSON — full raw results
json_path = os.path.join(EXPORT_DIR, 'results.json')
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'[1/7] results.json       {os.path.getsize(json_path):,} bytes')

In [ ]:
# 2 · CSV — flat table, one row per equation
rows = []
for eq in FEYNMAN_30:
    eid = eq['id']
    r   = all_results.get(eid, {})
    h   = r.get('hypatia', {})
    nn  = r.get('nn', {})
    he  = h.get('extrap_r2')
    rows.append({
        'equation_id':         eid,
        'name':                eq['name'],
        'domain':              eq['domain'],
        'hyp_train_r2':        h.get('train_r2'),
        'hyp_extrap_r2':       he,
        'hyp_time_s':          h.get('time_s'),
        'hyp_success':         (he is not None and he > 0.99),
        'nn_train_r2':         nn.get('train_r2'),
        'nn_extrap_r2':        nn.get('extrap_r2'),
        'nn_time_s':           nn.get('time_s'),
        'hyp_best_expression': h.get('best_expression', ''),
        'error':               r.get('error', ''),
    })

df_export = pd.DataFrame(rows)
csv_path = os.path.join(EXPORT_DIR, 'results.csv')
df_export.to_csv(csv_path, index=False)
print(f'[2/7] results.csv        {os.path.getsize(csv_path):,} bytes  ({len(df_export)} rows)')
df_export

In [ ]:
# 3 · LaTeX table (latex variable built in cell 12)
tex_path = os.path.join(EXPORT_DIR, 'table.tex')
with open(tex_path, 'w') as f:
    f.write(latex)
print(f'[3/7] table.tex          {os.path.getsize(tex_path):,} bytes')

In [ ]:
# 4 · HTML colour-coded table
def _bg(v, extrap=False):
    try: v = float(v)
    except: return ''
    if extrap:
        if v > 0.99: return 'background:#d4edda;color:#155724;font-weight:bold'
        if v < 0:    return 'background:#f8d7da;color:#721c24'
    return ''

html_rows = []
for eq in FEYNMAN_30:
    eid = eq['id']
    r   = all_results.get(eid, {})
    h   = r.get('hypatia', {})
    nn  = r.get('nn', {})
    he, ne, ht = h.get('extrap_r2'), nn.get('extrap_r2'), h.get('train_r2')
    win = he is not None and he > 0.99
    icon = '\u2713 Win' if win else '\u2717 Fail'
    ibg  = 'background:#d4edda' if win else 'background:#f8d7da'
    fmt  = lambda v: f'{v:.4f}' if v is not None else '\u2014'
    html_rows.append(
        f'<tr><td><code>{eid}</code></td><td>{eq["name"]}</td>'
        f'<td>{eq["domain"]}</td><td>{fmt(ht)}</td>'
        f'<td style="{_bg(he,True)}">{fmt(he)}</td>'
        f'<td style="{_bg(ne,True)}">{fmt(ne)}</td>'
        f'<td style="{ibg};font-weight:bold">{icon}</td></tr>'
    )

html_table_str = (
    '<!DOCTYPE html><html><head><meta charset="utf-8">'
    '<title>Feynman n=30 Results</title>'
    '<style>body{font-family:Arial,sans-serif;font-size:13px;margin:24px}'
    'table{border-collapse:collapse;width:100%}'
    'th,td{border:1px solid #ccc;padding:6px 10px;text-align:left}'
    'th{background:#343a40;color:white}'
    'tr:nth-child(even){background:#f9f9f9}'
    'code{font-size:12px}</style></head><body>'
    '<h2>Feynman Extrapolation \u2014 All n=30 Results</h2>'
    f'<p>HypatiaX vs Neural Network \u00b7 Full-budget (timeout={CFG["timeout"]}s, populations={CFG["populations"]})</p>'
    '<table><thead><tr>'
    '<th>ID</th><th>Name</th><th>Domain</th>'
    '<th>H Train R\u00b2</th><th>H Extrap R\u00b2</th>'
    '<th>NN Extrap R\u00b2</th><th>Result</th>'
    '</tr></thead><tbody>' +
    ''.join(html_rows) +
    '</tbody></table></body></html>'
)

html_table_path = os.path.join(EXPORT_DIR, 'table.html')
with open(html_table_path, 'w', encoding='utf-8') as f:
    f.write(html_table_str)
print(f'[4/7] table.html         {os.path.getsize(html_table_path):,} bytes')

In [ ]:
# 5 · stats.json — machine-readable summary
_hyp_all = [r.get('hypatia',{}).get('extrap_r2') for r in all_results.values()
             if r.get('hypatia',{}).get('extrap_r2') is not None]
_nn_all  = [r.get('nn',{}).get('extrap_r2') for r in all_results.values()
             if r.get('nn',{}).get('extrap_r2') is not None]
_succ_hyp = [v for v in _hyp_all if v > 0.99]
_succ_nn  = [r.get('nn',{}).get('extrap_r2')
              for r in all_results.values()
              if (r.get('hypatia',{}).get('extrap_r2') or -999) > 0.99
              and r.get('nn',{}).get('extrap_r2') is not None]

stats_out = {
    'n_equations': len(all_results),
    'n_successes': len(_succ_hyp),
    'success_rate': len(_succ_hyp) / len(all_results) if all_results else 0,
    'hyp_extrap_mean':   float(np.mean(_hyp_all))   if _hyp_all else None,
    'hyp_extrap_median': float(np.median(_hyp_all)) if _hyp_all else None,
    'nn_extrap_mean':    float(np.mean(_nn_all))    if _nn_all  else None,
    'nn_extrap_median':  float(np.median(_nn_all))  if _nn_all  else None,
}

if len(_hyp_all) >= 5 and len(_nn_all) >= 5:
    u1, p1 = scipy_stats.mannwhitneyu(_hyp_all, _nn_all, alternative='greater')
    stats_out['mw_all_U'] = float(u1)
    stats_out['mw_all_p'] = float(p1)
    stats_out['mw_all_significant'] = bool(p1 < 0.05)

if len(_succ_hyp) >= 3 and len(_succ_nn) >= 3:
    u2, p2 = scipy_stats.mannwhitneyu(_succ_hyp, _succ_nn, alternative='greater')
    stats_out['mw_succ_U'] = float(u2)
    stats_out['mw_succ_p'] = float(p2)
    stats_out['mw_succ_significant'] = bool(p2 < 0.05)

stats_path = os.path.join(EXPORT_DIR, 'stats.json')
with open(stats_path, 'w') as f:
    json.dump(stats_out, f, indent=2)
print(f'[5/7] stats.json         {os.path.getsize(stats_path):,} bytes')
print(json.dumps(stats_out, indent=2))

In [ ]:
# 6 · expressions.txt — best symbolic formulas found
lines = ['HypatiaX Best Symbolic Expressions\n', '=' * 50 + '\n']
successes = [(eid, r) for eid, r in all_results.items()
             if (r.get('hypatia',{}).get('extrap_r2') or -999) > 0.99]
failures  = [(eid, r) for eid, r in all_results.items()
             if (r.get('hypatia',{}).get('extrap_r2') or -999) <= 0.99]

lines.append(f'\nSUCCESSES ({len(successes)})\n' + '-' * 30 + '\n')
for eid, r in successes:
    h = r.get('hypatia', {})
    lines.append(f"{eid} ({r['name']})\n")
    lines.append(f"  extrap_R2 = {h.get('extrap_r2'):.4f}\n")
    lines.append(f"  expr      = {h.get('best_expression', 'N/A')}\n\n")

lines.append(f'FAILURES ({len(failures)})\n' + '-' * 30 + '\n')
for eid, r in failures:
    h = r.get('hypatia', {})
    he = h.get('extrap_r2')
    # Fix: Separate conditional logic from format specifier
    formatted_he = f"{he:.4f}" if he is not None else 'N/A'
    lines.append(f"{eid} ({r['name']})  extrap_R2 = {formatted_he}\n")

expr_path = os.path.join(EXPORT_DIR, 'expressions.txt')
with open(expr_path, 'w') as f:
    f.writelines(lines)
print(f'[6/7] expressions.txt    {os.path.getsize(expr_path):,} bytes')


In [ ]:
# 7 · report.html — full self-contained report
from datetime import datetime  # guard: may already be imported
n_eq   = len(all_results)
n_succ = len(_succ_hyp)
mw1_str = stats_out.get('mw_all_p') and (
    f"U={stats_out['mw_all_U']:.0f}, p={stats_out['mw_all_p']:.4e} "
    f"({'significant' if stats_out['mw_all_significant'] else 'not significant'})"
) or 'N/A'
mw2_str = stats_out.get('mw_succ_p') and (
    f"U={stats_out['mw_succ_U']:.0f}, p={stats_out['mw_succ_p']:.4e} "
    f"({'significant' if stats_out['mw_succ_significant'] else 'not significant'})"
) or 'N/A'

succ_rows = ''.join(
    f'<tr><td><code>{eid}</code></td><td>{r["name"]}</td>'
    f'<td>{r.get("hypatia",{}).get("extrap_r2",0):.4f}</td>'
    f'<td style="font-family:monospace;font-size:11px">{r.get("hypatia",{}).get("best_expression","N/A")}</td></tr>'
    for eid, r in all_results.items()
    if (r.get('hypatia',{}).get('extrap_r2') or -999) > 0.99
)

report = f"""<!DOCTYPE html>
<html><head><meta charset=\"utf-8\">
<title>RF-09 Feynman Extrapolation Report</title>
<style>
  body{{font-family:Arial,sans-serif;margin:40px;max-width:900px;color:#222}}
  h1{{border-bottom:3px solid #343a40;padding-bottom:8px}}
  h2{{color:#343a40;margin-top:32px}}
  table{{border-collapse:collapse;width:100%;margin:10px 0}}
  th,td{{border:1px solid #ccc;padding:7px 12px;text-align:left}}
  th{{background:#343a40;color:#fff}}
  tr:nth-child(even){{background:#f5f5f5}}
  .box{{background:#f0f4ff;border-left:4px solid #4a6fa5;padding:12px 18px;margin:12px 0;border-radius:4px}}
  .claim{{background:#fff8e1;border-left:4px solid #f0ad4e;padding:12px 18px;margin:16px 0;border-radius:4px}}
  .pass{{background:#d4edda;color:#155724;font-weight:bold}}
  footer{{margin-top:48px;font-size:11px;color:#999;border-top:1px solid #eee;padding-top:8px}}
</style></head><body>
<h1>RF-09 &#8212; Feynman Extrapolation: n=30 Full Report</h1>
<p><strong>HypatiaX Research</strong> &middot; {datetime.now().strftime('%Y-%m-%d %H:%M')} &middot;
timeout={CFG['timeout']}s &middot; populations={CFG['populations']} &middot; iterations={CFG['iterations']}</p>

<h2>Summary</h2>
<div class=\"box\">
HypatiaX successes (extrap R&sup2; &gt; 0.99): <strong>{n_succ}/{n_eq} ({n_succ/n_eq*100:.1f}%)</strong><br>
HypatiaX extrap R&sup2; &mdash; mean: <strong>{'%.4f' % np.mean(_hyp_all) if _hyp_all else 'N/A'}</strong> &nbsp; median: <strong>{'%.4f' % np.median(_hyp_all) if _hyp_all else 'N/A'}</strong><br>
NN extrap R&sup2; &mdash; mean: <strong>{'%.4f' % np.mean(_nn_all) if _nn_all else 'N/A'}</strong> &nbsp; median: <strong>{'%.4f' % np.median(_nn_all) if _nn_all else 'N/A'}</strong>
</div>

<h2>Mann-Whitney tests</h2>
<table>
  <tr><th>Test</th><th>n</th><th>Result</th></tr>
  <tr><td>Test 1 &mdash; all n={n_eq} (primary claim)</td><td>{n_eq}</td><td>{mw1_str}</td></tr>
  <tr><td>Test 2 &mdash; successes only (sub-claim)</td><td>{n_succ}</td><td>{mw2_str}</td></tr>
</table>

<div class=\"claim\">
<strong>Suggested paper claim (replace current n=14 framing):</strong><br>
&#8220;HypatiaX finds a symbolic formula on <strong>{n_succ}/{n_eq}</strong> Feynman equations.
On those {n_succ} equations, extrapolation significantly outperforms the NN baseline
(Mann-Whitney {mw2_str}).
On the remaining {n_eq - n_succ} equations where HypatiaX failed to find a formula,
PySR&#8217;s best approximation is also reported.&#8221;
</div>

<h2>Successful equations</h2>
<table>
  <tr><th>ID</th><th>Name</th><th>Extrap R&sup2;</th><th>Best expression</th></tr>
  {succ_rows}
</table>

<h2>Full results table</h2>
{html_table_str}

<footer>Generated by exp2_feynman_extrap_colab.ipynb &middot; HypatiaX Research &middot; RF-09</footer>
</body></html>"""

report_path = os.path.join(EXPORT_DIR, 'report.html')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)
print(f'[7/7] report.html        {os.path.getsize(report_path):,} bytes')

In [ ]:
import zipfile, os
zip_path = os.path.join(OUTPUT_DIR, 'hypatia_exp2_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(EXPORT_DIR):
        zf.write(os.path.join(EXPORT_DIR, fname), fname)
print(f'Zip: {zip_path}  ({os.path.getsize(zip_path):,} bytes)')
for fname in sorted(os.listdir(EXPORT_DIR)):
    print(f"  {fname:<30} {os.path.getsize(os.path.join(EXPORT_DIR, fname)):>8,} bytes")
print()
print('All outputs in /kaggle/working/')
print('Download: Output tab (right sidebar) -> Download all')